<a href="https://colab.research.google.com/github/jose02082000/Curso-programacion-cientifica/blob/main/ejercicios/taller-evaluativo-03-numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Evaluativo 03 — NumPy
**Estudiante:** Jose David Muñoz Restrepo
**Materia:** Programación Científica  
**Fecha:** Octubre de 2026

## Ejercicio 1 — Crear los arreglos y leer su etiqueta
En este ejercicio convertimos la tabla de lecturas crudas en un arreglo bidimensional de NumPy (5 filas, 7 columnas), extraemos sus propiedades estructurales mediante sus atributos (`shape`, `ndim`, `size`, `dtype`) y creamos los arreglos para los días de la semana y los parámetros de calibración (pendientes e interceptos) de cada sensor.

In [1]:
import numpy as np

lecturas_crudas = np.array([
    [10.0, 14.5, 22.0, 30.5, 28.0, 18.0,  9.5],   # SBC-A
    [12.0, 20.0, 35.0, 48.0, 41.0, 25.0, 14.0],   # SBC-B
    [ 8.0, 11.0, 19.0, 27.0, 33.0, 21.0,  6.0],   # SBC-C
    [15.0, 24.0, 31.0, 44.0, 52.0, 36.0, 17.0],   # SBC-D
    [ 5.0,  9.0, 16.0, 22.0, 45.0, 30.0,  4.0],   # SBC-E
])

forma = lecturas_crudas.shape
dimensiones = lecturas_crudas.ndim
total = lecturas_crudas.size
tipo = lecturas_crudas.dtype

dias = np.arange(1, 8)
pendientes = np.array([1.2, 0.9, 1.1, 1.0, 1.3])
interceptos = np.array([-0.5, 1.0, -2.0, 0.5, -1.5])

print(f"Forma: {forma}")
print(f"Dimensiones: {dimensiones}")
print(f"Total de elementos: {total}")
print(f"Tipo de dato: {tipo}")
print(f"Días: {dias}")
print(f"Pendientes: {pendientes}")
print(f"Interceptos: {interceptos}")

Forma: (5, 7)
Dimensiones: 2
Total de elementos: 35
Tipo de dato: float64
Días: [1 2 3 4 5 6 7]
Pendientes: [1.2 0.9 1.1 1.  1.3]
Interceptos: [-0.5  1.  -2.   0.5 -1.5]


## Ejercicio 2 — Señalar celdas y recortar bloques
Utilizamos indexación y slicing sobre el arreglo `lecturas_crudas` para extraer celdas individuales (incluyendo indexación negativa), filas completas, columnas completas y un subbloque específico de la matriz.

In [2]:
lectura_c_dia4 = lecturas_crudas[2, 3]
ultimo_primer = lecturas_crudas[-1, 0]
dia5 = lecturas_crudas[:, 4]
semana_b = lecturas_crudas[1, :]
bloque = lecturas_crudas[1:4, 2:6]

print(f"Lectura SBC-C día 4: {lectura_c_dia4}")
print(f"Último sensor día 1: {ultimo_primer}")
print(f"Lecturas día 5: {dia5}")
print(f"Lecturas semana SBC-B: {semana_b}")
print(f"Bloque extraído:\n{bloque}")
print(f"Forma del bloque: {bloque.shape}")

Lectura SBC-C día 4: 27.0
Último sensor día 1: 5.0
Lecturas día 5: [28. 41. 33. 52. 45.]
Lecturas semana SBC-B: [12. 20. 35. 48. 41. 25. 14.]
Bloque extraído:
[[35. 48. 41. 25.]
 [19. 27. 33. 21.]
 [31. 44. 52. 36.]]
Forma del bloque: (3, 4)


## Ejercicio 3 — Limpiar una campaña con máscaras booleanas
A partir de la serie de mediciones de campaña de SBC-A, construimos máscaras booleanas para identificar datos faltantes (centinela -999.0) y lecturas válidas/útiles en el rango [0.0, 500.0]. Finalmente, filtramos el arreglo y calculamos su promedio sin utilizar bucles.

In [3]:
campana_sbc_a = np.array([10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0])

faltantes = (campana_sbc_a == -999.0)
es_util = (campana_sbc_a >= 0.0) & (campana_sbc_a <= 500.0)

lecturas_utiles = campana_sbc_a[es_util]
promedio_utiles = lecturas_utiles.mean()

print(f"Cantidad de lecturas faltantes: {faltantes.sum()}")
print(f"Lecturas útiles: {lecturas_utiles}")
print(f"Promedio de lecturas útiles: {promedio_utiles}")

Cantidad de lecturas faltantes: 2
Lecturas útiles: [10. 25. 40.  8. 52.]
Promedio de lecturas útiles: 27.0


## Explicación del Broadcasting en el Ejercicio 4
La operación directa `lecturas_crudas * pendientes` produce un error de tipo `ValueError` debido a que las formas de los arreglos no son compatibles para el alineamiento automático (broadcasting). `lecturas_crudas` tiene una forma bidimensional de (5, 7) y `pendientes` es un arreglo unidimensional de forma (5,). Según las reglas de broadcasting de NumPy, los arreglos se alinean de derecha a izquierda: el eje de tamaño 5 de `pendientes` se alinearía con las 7 columnas de `lecturas_crudas`, lo cual es incompatible. Para operar cada fila de la matriz con su correspondiente pendiente, debemos transformar `pendientes` a una matriz columna de forma (5, 1).

In [4]:
pendientes_col = pendientes.reshape(5, 1)
interceptos_col = interceptos.reshape(5, 1)

calibradas = lecturas_crudas * pendientes_col + interceptos_col

campana_calibrada = lecturas_utiles * 1.2 - 0.5
promedio_calibrado_sbc_a = campana_calibrada.mean()

print(f"Forma de la matriz calibrada: {calibradas.shape}")
print(f"Fila calibrada SBC-A: {calibradas[0]}")
print(f"Campaña calibrada SBC-A: {campana_calibrada}")
print(f"Promedio calibrado SBC-A: {promedio_calibrado_sbc_a}")

Forma de la matriz calibrada: (5, 7)
Fila calibrada SBC-A: [11.5 16.9 25.9 36.1 33.1 21.1 10.9]
Campaña calibrada SBC-A: [11.5 29.5 47.5  9.1 61.9]
Promedio calibrado SBC-A: 31.9


## Ejercicio 5 — Resumir la semana con estadística por eje
Calculamos estadísticas descriptivas globales, por sensor (`axis=1`) y por día (`axis=0`) sobre la matriz `calibradas`. También obtenemos los valores máximos por sensor y el promedio de aquellos datos que superan el umbral de 35.4 µg/m³.

In [5]:
promedio_general = calibradas.mean()
promedio_por_sensor = calibradas.mean(axis=1)
promedio_por_dia = calibradas.mean(axis=0)
maximo_por_sensor = calibradas.max(axis=1)

mascara_sobre_limite = calibradas > 35.4
media_sobre_limite = calibradas[mascara_sobre_limite].mean()

print(f"Promedio general: {np.round(promedio_general, 2)}")
print(f"Promedio por sensor: {np.round(promedio_por_sensor, 2)}")
print(f"Promedio por día: {np.round(promedio_por_dia, 2)}")
print(f"Máximo por sensor: {np.round(maximo_por_sensor, 2)}")
print(f"Media sobre el límite (35.4): {np.round(media_sobre_limite, 2)}")

Promedio general: 24.11
Promedio por sensor: [22.21 26.07 17.64 31.79 22.83]
Promedio por día: [10.12 16.14 25.62 35.92 42.96 27.94 10.06]
Máximo por sensor: [36.1 44.2 34.3 52.5 57. ]
Media sobre el límite (35.4): 43.28


## Ejercicio 6 — Funciones universales y reemplazo con máscara
Utilizamos funciones universales como `np.maximum` para calcular excedentes de emisión sobre el umbral de 35.4 µg/m³, `np.abs` para distancias absolutas y creamos una copia aislada de los datos (`en_alerta`) donde se reemplazan las lecturas normales por 0.0 mediante una máscara.

In [6]:
excedente = np.maximum(calibradas - 35.4, 0)
excedente_por_sensor = np.round(excedente.sum(axis=1), 1)

desvio_sensor_b = np.round(np.abs(calibradas[1] - 35.4), 1)

en_alerta = calibradas.copy()
en_alerta[en_alerta <= 35.4] = 0.0

celdas_alerta_cont = (en_alerta > 0).sum()

print(f"Excedente por sensor: {excedente_por_sensor}")
print(f"Desvío de SBC-B al límite: {desvio_sensor_b}")
print(f"Cantidad de celdas en alerta (> 0): {celdas_alerta_cont}")

Excedente por sensor: [ 0.7 11.3  0.  27.3 23.7]
Desvío de SBC-B al límite: [23.6 16.4  2.9  8.8  2.5 11.9 21.8]
Cantidad de celdas en alerta (> 0): 8


## Ejercicio 7 — Integración: el informe de la red
Clasificamos las 35 lecturas calibradas dentro de las cuatro categorías oficiales de calidad del aire utilizando máscaras booleanas exhaustivas y mutuamente excluyentes. Adicionalmente, identificamos el día pico de contaminación combinando una máscara sobre el vector `promedio_por_dia`.

In [7]:
buenas = int((calibradas <= 12.0).sum())
moderadas = int(((calibradas > 12.0) & (calibradas <= 35.4)).sum())
daninas_sensibles = int(((calibradas > 35.4) & (calibradas <= 55.4)).sum())
daninas_salud = int((calibradas > 55.4).sum())

mascara_pico = (promedio_por_dia == promedio_por_dia.max())
dia_pico = dias[mascara_pico][0]
valor_dia_pico = np.round(promedio_por_dia[mascara_pico][0], 1)

print(f"Lecturas por categoría: Buena={buenas}, Moderada={moderadas}, Dañina para grupos sensibles={daninas_sensibles}, Dañina para la salud={daninas_salud}")
print(f"Día más contaminado: {dia_pico} con {valor_dia_pico} ug/m3")

Lecturas por categoría: Buena=9, Moderada=18, Dañina para grupos sensibles=7, Dañina para la salud=1
Día más contaminado: 5 con 43.0 ug/m3


## Verificación Final
Celda de prueba automatizada para confirmar la corrección de arreglos, formas y valores procesados.

In [8]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert lecturas_crudas.shape == (5, 7) and lecturas_crudas.ndim == 2, "revise lecturas_crudas del Ejercicio 1"
assert forma == (5, 7) and dimensiones == 2 and total == 35 and tipo == np.float64, "revise los atributos del Ejercicio 1"
assert np.array_equal(dias, np.arange(1, 8)), "dias debe ir del 1 al 7 (el 7 incluido)"
assert np.allclose(pendientes, [1.2, 0.9, 1.1, 1.0, 1.3]) and np.allclose(interceptos, [-0.5, 1.0, -2.0, 0.5, -1.5]), "revise pendientes e interceptos"
assert lectura_c_dia4 == 27.0 and ultimo_primer == 5.0, "revise la indexación del Ejercicio 2"
assert np.array_equal(dia5, [28.0, 41.0, 33.0, 52.0, 45.0]) and semana_b.shape == (7,), "revise dia5 y semana_b del Ejercicio 2"
assert bloque.shape == (3, 4) and bloque[0, 0] == 35.0, "revise el bloque del Ejercicio 2"
assert int(faltantes.sum()) == 2 and lecturas_utiles.shape == (5,), "revise las máscaras del Ejercicio 3"
assert abs(promedio_utiles - 27.0) < 1e-9, "revise promedio_utiles del Ejercicio 3"
assert calibradas.shape == (5, 7), "calibradas debe tener forma (5, 7)"
assert np.allclose(calibradas[0], [11.5, 16.9, 25.9, 36.1, 33.1, 21.1, 10.9]), "revise la calibración del Ejercicio 4"
assert lecturas_crudas[0, 0] == 10.0, "lecturas_crudas no debe modificarse"
assert abs(promedio_calibrado_sbc_a - 31.9) < 1e-9, "revise promedio_calibrado_sbc_a del Ejercicio 4"
assert abs(promedio_general - 24.1086) < 1e-4, "revise promedio_general del Ejercicio 5"
assert np.allclose(np.round(promedio_por_sensor, 2), [22.21, 26.07, 17.64, 31.79, 22.83]), "revise promedio_por_sensor (axis=1)"
assert np.allclose(np.round(promedio_por_dia, 2), [10.12, 16.14, 25.62, 35.92, 42.96, 27.94, 10.06]), "revise promedio_por_dia (axis=0)"
assert np.allclose(maximo_por_sensor, [36.1, 44.2, 34.3, 52.5, 57.0]), "revise maximo_por_sensor"
assert abs(media_sobre_limite - 43.275) < 1e-9, "revise media_sobre_limite"
assert np.allclose(excedente_por_sensor, [0.7, 11.3, 0.0, 27.3, 23.7]), "revise excedente_por_sensor del Ejercicio 6"
assert np.allclose(desvio_sensor_b, [23.6, 16.4, 2.9, 8.8, 2.5, 11.9, 21.8]), "revise desvio_sensor_b"
assert en_alerta.shape == (5, 7) and int((en_alerta > 0).sum()) == 8, "revise en_alerta"
assert abs(calibradas[0, 3] - 36.1) < 1e-9, "calibradas no debe modificarse al crear en_alerta"
assert (buenas, moderadas, daninas_sensibles, daninas_salud) == (9, 18, 7, 1), "revise los conteos del Ejercicio 7"
assert buenas + moderadas + daninas_sensibles + daninas_salud == calibradas.size, "los cuatro conteos deben sumar 35"
assert dia_pico == 5, "revise dia_pico del Ejercicio 7"
print("Verificación completada sin errores.")

Verificación completada sin errores.
